# Final Project Notebook

Use the follow cells prompts to complete the final project for the course. Everything you need should be present in this notebook or previous notebooks we've used in class. You can work together as needed. 

 - You will need to name your own dataset and use that name throughout
 - There are sections where you need to make changes the code and insert new code this will be noted in the code provided
 - You may get frustrated along the way, this is totally normal, just remember even small changes to the code make a huge difference. 

## Question Fork the Repository
i. Include a screenshot of the forked repo in your GitHub account

To fork the repository:
1. Go to https://github.com/NovaVolunteer/ds1001_final
2. Click the "Fork" button in the top right corner
3. The repo will be forked to your GitHub account
4. Take a screenshot of your forked repository

### You should now be able to open your cloned repo in google collab, use the code below. 

### It's very helpful to have the variable inspector open while you go through this process. To do so go to tools>command palette>show variable inspector

### It's also helpful to open up the folder tree on the left menu bar. Just click on the folder icon and then the ds1001_final folder. The data is located in the data folder in the processed sub-folder. 

In [27]:
!git clone "https://github.com/username/repository.git"
# This script clones a GitHub repository using Git command line tool. 
# Insert the path to your desired repository in place of the URL.

Cloning into 'repository'...
remote: Repository not found.
fatal: repository 'https://github.com/username/repository.git/' not found


## Systems

In [28]:
# Activate the finalproj environment
!source ../finalproj/bin/activate

In [29]:
### You can use this command to list all the packages in your environment
!pip list

Package                   Version
------------------------- -----------
anyio                     4.11.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.0.5
attrs                     25.4.0
babel                     2.17.0
beautifulsoup4            4.14.2
bleach                    6.3.0
certifi                   2025.11.12
cffi                      2.0.0
charset-normalizer        3.4.4
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
debugpy                   1.8.17
decorator                 5.2.1
defusedxml                0.7.1
executing                 2.2.1
fairlearn                 0.13.0
fastjsonschema            2.21.2
fonttools                 4.60.1
fqdn                      1.5.1
gitdb                     4.0.12
GitPython                 3.1.46
h11                       0.16.0
httpcore     

In [30]:
!pip install fairlearn

#You'll likely need to install the fairlearn packages, if not already installed.
#Are there additional packages to install? (Cross check with the list above to 
# ensure all packages are installed)

### Check !pip list again to confirm installations

In [37]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import fairlearn.metrics
from fairlearn.metrics import MetricFrame
from fairlearn.metrics import count, true_positive_rate, false_positive_rate, selection_rate, demographic_parity_ratio

## Design: Data prep and exploration 

In [35]:
df = pd.read_csv('../data/processed/bank_final.csv') # the data is the data folder, 
#you'll need to use the correct path to the dataset. 

# How many rows are in the dataframe? How many columns?
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 43628
Number of columns: 13


In [38]:
# Explore the variables a bit more, create histograms for the numerics values and bar charts for the categorical.
# Histograms for numeric variables
numeric_columns = df.select_dtypes(include=['number']).columns
for col in numeric_columns: 
    plt.figure(figsize=(10, 6))
    sns.histplot(df[col], kde=True)
    plt.title(f'Histogram of {col}')
    plt.show()  

In [39]:
# Bar charts for categorical variables
categorical_columns = df.select_dtypes(include=['object', 'category']).columns
for col in categorical_columns:
    plt.figure(figsize=(10, 6))
    sns.countplot(x=df[col])
    plt.title(f'Bar Chart of {col}')
    plt.xticks(rotation=45)
    plt.show()

In [40]:
# How many numeric columns are in the data set?
num_numeric_columns = df.select_dtypes(include=['number']).shape[1]
print(f"Number of numeric columns: {num_numeric_columns}")

Number of numeric columns: 5


In [41]:
# Normalization
numeric_cols = df.select_dtypes(include=['number']).columns.drop('signed up')
scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [42]:
# Likely need to convert categorical columns to category dtype
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype('category')

In [43]:
# Save sensitive features before dummying
sensitive_race = df['race']
sensitive_gender = df['gender']

# Creating dummy variables, make sure the variables that need to be converted to dummies are categorical, not numeric.
# This might require you to convert some columns to categorical first using astype('category')
df = pd.get_dummies(df, drop_first=True)

In [44]:
# Display missing data using the isnull function, is there any missing data?
print(df.isnull().sum())

age                    0
balance                0
duration               0
contactndays           0
signed up              0
job_blue-collar        0
job_entrepreneur       0
job_housemaid          0
job_management         0
job_retired            0
job_self-employed      0
job_services           0
job_student            0
job_technician         0
job_unemployed         0
job_unknown            0
marital_married        0
marital_single         0
education_secondary    0
education_tertiary     0
education_unknown      0
race_black             0
race_hispanic          0
race_white             0
default_yes            0
housing_yes            0
contact_telephone      0
contact_unknown        0
gender_m               0
dtype: int64


In [ ]:
# remove missing values if needed
"xx" = "xx".dropna()

In [45]:
# Scatterplot between two continuous variables
plt.figure(figsize=(10, 6))
sns.scatterplot(x='age', y='balance', data=df)  # Replace 'Variable1' and 'Variable2' with your column names
plt.title('Scatterplot of age vs balance')
plt.savefig('scatterplot.png')  # Save the scatterplot image
plt.show()

In [46]:
# Density chart of a continuous variable
plt.figure(figsize=(10, 6))
sns.kdeplot(df['age'], fill=True)  # Replace 'ContinuousVariable' with your column name
plt.title('Density Chart of age')
plt.show()

In [47]:
#Correlation matrix, make sure to only include numeric variables
num_values = df.select_dtypes(include=['number'])
correlation_matrix = num_values.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.show()

## Analytics: Build a model and Tune it for best Best Performance

In [48]:
# What is the ‘target’ of a model and what is the prevalence of the target in your dataset? Remember prevalence 
# is the proportion of records that take on the value of interest for the target variable, usually the positive class.
target_prevalence = df['signed up'].mean()  # Replace 'TargetVariable' with your target column name
print(f'Target Prevalence: {target_prevalence}')

Target Prevalence: 0.11630145777940772


In [49]:
# Divide the dataset into features and target
target = df['signed up']  # Replace 'TargetVariable' with your actual target column name and "xx" with your dataframe name
features = df.drop(columns=['signed up']) # Drop the target column from features

In [50]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Get sensitive features for test set
sensitive_race_test = sensitive_race.iloc[X_test.index]
sensitive_gender_test = sensitive_gender.iloc[X_test.index]

In [51]:
# Include your table for the 10 values of k you tried and the corresponding accuracies.

accuracy_results = {}

# Replace x with your desired range values, explain what is happening in this loop
for k in range(1, 11):  
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train, y_train)
    accuracy = knn_model.score(X_test, y_test)
    accuracy_results[k] = accuracy

print("Accuracy results:")
for k, acc in accuracy_results.items():
    print(f"k={k}: {acc:.4f}")

best_k = max(accuracy_results, key=accuracy_results.get)
print(f"Best k: {best_k} with accuracy {accuracy_results[best_k]:.4f}")

Accuracy results:
k=1: 0.8589
k=2: 0.8844
k=3: 0.8767
k=4: 0.8809
k=5: 0.8796
k=6: 0.8826
k=7: 0.8802
k=8: 0.8822
k=9: 0.8828
k=10: 0.8831
Best k: 2 with accuracy 0.8844


In [52]:
#graph of accuracy vs k values
plt.figure(figsize=(10, 6))
plt.plot(list(accuracy_results.keys()), list(accuracy_results.values()), marker='o')
plt.title('KNN Accuracy vs K Values')
plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Accuracy')
plt.xticks(list(accuracy_results.keys()))
plt.grid()
plt.show()

In [53]:
# using the hyperparameter k that gave the best accuracy, rerun the model and generate 
# predictions on the test set. Explain why you choose this k value.
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)

## Value: Evaluation and Protected Classes

In [54]:
# create a confusion matrix for your model's predictions. 
# What does the confusion matrix tell you about your model's performance?
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm)
# disp.plot()
# plt.title('Confusion Matrix')
# plt.show()

Confusion Matrix:
[[7596   96]
 [ 913  121]]


In [55]:
#We already have a model above using KNN so we can use the results to compute fairness metrics

# Compute fairness metrics using Fairlearn

my_metrics = {
    'true positive rate' : true_positive_rate,
    'false positive rate' : false_positive_rate,
    'selection rate' : selection_rate,
    'count' : count
}
# Construct a MetricFrame for race
mf_race = MetricFrame(
    metrics=my_metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_race_test  # Replace with your first protected class
)

# Construct a MetricFrame for gender
mf_gender = MetricFrame(
    metrics=my_metrics,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sensitive_gender_test  # Replace second protected class 
)

In [56]:
mf_race.by_group #What do the results show? Change the mf_race with each subgroup and report the findings. This means
# you should run this cell multiple times, once for each of the levels in the race variable.

,true positive rate,false positive rate,selection rate,count
race,,,,
asian,0.088462,0.012712,0.021881,2148.0
black,0.150862,0.011117,0.025780,2211.0
hispanic,0.111538,0.011665,0.023765,2146.0
white,0.120567,0.014440,0.027915,2221.0


In [57]:
mf_gender.by_group #What do the results show? There's only two groups here so we don't need to change anything. 
# in the metric frame.

,true positive rate,false positive rate,selection rate,count
gender,,,,
f,0.117117,0.012645,0.025971,4351.0
m,0.116910,0.012320,0.023771,4375.0


In [58]:
# Derived fairness metrics. Be sure you understand the scale and meaning of these. Here we are calculating the 
# two fairness ratios using the gender_m feature, which is bi-variate. What do the results show, is the model more or 
# less fair with this grouping?

dpr_gender = fairlearn.metrics.demographic_parity_ratio(y_test, y_pred, sensitive_features=sensitive_gender_test)
print("Demographic Parity ratio:\t", dpr_gender)

eodds_gender = fairlearn.metrics.equalized_odds_ratio(y_test, y_pred, sensitive_features=sensitive_gender_test)
print("Equalized Odds ratio:\t\t", eodds_gender)

Demographic Parity ratio:	 0.915305183312263
Equalized Odds ratio:		 0.9743326488706366


In [59]:
# Derived fairness metrics. Be sure you understand the scale and meaning of these. Here we are calculating the 
# the same features above only using a filtered search to pull in all the possibilities of features
# starting with "race". What do the results show, is the model more or less fair with this grouping?

dpr_race = fairlearn.metrics.demographic_parity_ratio(y_test, y_pred, sensitive_features=sensitive_race_test)
print("Demographic Parity ratio:\t", dpr_race)

eodds_race = fairlearn.metrics.equalized_odds_ratio(y_test, y_pred, sensitive_features=sensitive_race_test)
print("Equalized Odds ratio:\t\t", eodds_race)

Demographic Parity ratio:	 0.7838274163512946
Equalized Odds ratio:		 0.5863736263736264


In [ ]:
#Optional code to add, commit, and push changes to your GitHub repository
!git add .
!git commit -m "Insert Message Here" # This will commit your changes to git. 
!git push # This will push your changes to back to your remote repository on GitHub.